# ml-inference: full-scale training on a free Colab T4

Runs the whole training and evaluation path of
[jasonjesuraja06/ml-inference](https://github.com/jasonjesuraja06/ml-inference) at full
scope on one free-tier T4, by calling the repository's own scripts with no edits. It
downloads the data, builds the splits, runs the lint and test gates, trains all three
models, freezes each run's raw output under `results/`, and prints a markdown results
table.

**Nothing in this notebook is precomputed.** Every cell ships with its outputs cleared.
The numbers appear once it is run, and they come from the same scripts the README
documents, so a table produced here can be compared line for line against the CPU
numbers already in the README.

Why it exists: the README's numbers were measured on an Apple M4 Pro with no CUDA GPU.
That host caps how much training fits in a sitting. This notebook is the GPU path, and
it runs the identical commands so the only variable is the accelerator.

Runtime > Change runtime type > T4 GPU. Expect 60 to 110 minutes end to end on the free
tier, most of it in section 5.

## 1. GPU check

Confirms a CUDA device is visible and names it. Every number this notebook prints must
carry that name, because a different GPU is a different measurement.
Expected wall clock: under 1 minute.

In [ ]:
import json
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import torch


def sh(cmd, env=None, echo=True):
    """Run a command, stream its output, and return (exit code, captured lines)."""
    merged = {**os.environ, **(env or {})}
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=merged
    )
    lines = []
    for line in proc.stdout:
        lines.append(line.rstrip("\n"))
        if echo:
            print(line, end="")
    proc.wait()
    return proc.returncode, lines


def must(cmd, env=None):
    rc, lines = sh(cmd, env=env)
    assert rc == 0, f"exit code {rc}: {' '.join(cmd)}"
    return lines


try:
    rc, _ = sh(["nvidia-smi"])
except FileNotFoundError:
    rc = 1
assert rc == 0, "nvidia-smi failed: switch to a GPU runtime"
assert torch.cuda.is_available(), "torch sees no CUDA device: switch to a GPU runtime"
GPU_NAME = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
print(f"gpu: {GPU_NAME}")
print(f"memory: {props.total_memory / 1e9:.1f} GB")
print(f"capability: sm{props.major}{props.minor}")
if "T4" not in GPU_NAME:
    print(f"warning: this is not a T4; label every reported number with {GPU_NAME!r}")

## 2. Setup

Clones the repository and installs it editable. A reused Colab runtime keeps its old
checkout, so an existing clone is fast-forwarded rather than left stale, and the commit
actually being run is printed.

If pip reports that the runtime must be restarted, restart it and re-run from this cell.
The clone is idempotent and nothing before this point produced a number.
Expected wall clock: 3 to 6 minutes.

In [ ]:
BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = BASE / "ml-inference"
os.environ["HF_HOME"] = str(BASE / "hf_home")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not (REPO / "pyproject.toml").exists():
    must(["git", "clone", "https://github.com/jasonjesuraja06/ml-inference", str(REPO)])
else:
    must(["git", "-C", str(REPO), "pull", "--ff-only"])
COMMIT = must(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], )[0].strip()
sh(["git", "-C", str(REPO), "log", "--oneline", "-1"])
os.chdir(REPO)

must([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"])
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

# The repo's scripts and modules resolve imports through PYTHONPATH, exactly as the
# Makefile sets it. Every training command below inherits this.
RUN_ENV = {"PYTHONPATH": f"{REPO / 'src'}:{REPO}"}
print("commit:", COMMIT)

## 3. Data

Runs `scripts/download_data.py` and `scripts/build_splits.py` unchanged. The download is
roughly 200 MB and expands to about 700 MB. `build_splits.py` keeps only the `target == 1`
rows of DiverseVul, because the dataset attaches a fixing commit's CWE to both the
vulnerable and the patched function, and prints how many rows survive.

`scripts/cwe_support.py` then prints the CWE frequency distribution. `TOP_K_CWES = 10` is
fixed in `src/ml_inference/config.py` before any model is trained; this cell is what makes
that cut checkable rather than asserted.
Expected wall clock: 5 to 12 minutes, mostly the download.

In [ ]:
must([sys.executable, "scripts/download_data.py"], env=RUN_ENV)

In [ ]:
must([sys.executable, "scripts/build_splits.py"], env=RUN_ENV)
print()
must([sys.executable, "scripts/cwe_support.py"], env=RUN_ENV)
print()
print(json.dumps(json.loads((REPO / "data/splits/manifest.json").read_text()), indent=2))

## 4. Gates

Lint and the test suite, both run before any training. A failure here stops the notebook,
because a number produced by code that does not pass its own gates is not worth reporting.
Expected wall clock: 1 to 2 minutes.

In [ ]:
GATE_RESULTS = []
for label, cmd in [
    ("ruff", [sys.executable, "-m", "ruff", "check", "src", "api", "tests", "scripts", "bench"]),
    ("pytest", [sys.executable, "-m", "pytest", "-q"]),
]:
    t0 = time.monotonic()
    rc, _ = sh(cmd, env=RUN_ENV)
    took = time.monotonic() - t0
    assert rc == 0, f"gate failed: {label} (exit {rc})"
    GATE_RESULTS.append((label, " ".join(cmd), took))
    print(f"gate passed: {label} ({took:.0f}s)")

## 5. Training

Three runs, in the order the README reports them. Each uses the whole split it is given
and the epoch count the README documents, so this is a like-for-like rerun of the CPU
numbers on a GPU rather than a different experiment:

| Run | Scope | Epochs |
|---|---|---|
| `train_baseline` | all 9,854 DiverseVul train rows, 2,415-row holdout | 3 |
| `train_improved` | the same rows plus minority-class augmentation | 3 |
| `train_devign` | all 21,854 CodeXGLUE Devign train rows, 2,732-row test | 2 |

Three epochs sits in the 2 to 4 range Devlin et al. recommend for fine-tuning
(BERT, NAACL-HLT 2019, section A.3), chosen against that range rather than searched, and
held identical across the two DiverseVul arms. Those two arms still differ in backbone and
learning rate as well as in the imbalance handling, so neither run isolates a single
factor; `docs/benchmarks.md` says so too. The Devign run's 2 epochs is what fit the CPU
budget of the reference run; section 7 additionally runs the CodeXGLUE reference recipe,
which a T4 can afford and the CPU host could not.

`scripts/collect_result.py` freezes each run into `results/<name>/` with its metrics JSON,
per-class table, confusion matrix, wall clock, stdout, and the exact command.
Expected wall clock: 25 to 50 minutes for all three on a T4.

In [ ]:
def train(name, module, epochs, report, extra_env=None):
    """Run one training module at full scope, then freeze its output under results/."""
    env = {**RUN_ENV, "EPOCHS": str(epochs), **(extra_env or {})}
    prefix = " ".join(f"{k}={v}" for k, v in sorted((extra_env or {}).items()))
    command = f"EPOCHS={epochs} {prefix} python -m {module}".replace("  ", " ").strip()
    log = BASE / f"{name}.log"

    t0 = time.monotonic()
    rc, lines = sh([sys.executable, "-m", module], env=env)
    took = time.monotonic() - t0
    log.write_text("\n".join(lines) + "\n")
    assert rc == 0, f"{name} failed with exit {rc}"

    must([
        sys.executable, "scripts/collect_result.py",
        "--name", name,
        "--report", report,
        "--command", command,
        "--log", str(log),
    ], env=RUN_ENV)
    print(f"{name}: {took / 60:.1f} min")
    return took

In [ ]:
train(
    "cwe_baseline",
    "ml_inference.train_baseline",
    epochs=3,
    report="bench/reports/baseline_holdout_metrics.json",
)

In [ ]:
train(
    "cwe_improved",
    "ml_inference.train_improved",
    epochs=3,
    report="bench/reports/improved_holdout_metrics.json",
    extra_env={"NO_AUTO_LABELS": "1"},
)

In [ ]:
train(
    "devign_codebert",
    "ml_inference.train_devign",
    epochs=2,
    report="bench/reports/devign_metrics.json",
)

## 6. Results table

`scripts/results_table.py` reads only the frozen files under `results/` and the
literature numbers transcribed in `results/published_baselines.json`. It computes
nothing, so what it prints and what is committed carry the same digits.

The output below is markdown. Paste it into the README, replacing the corresponding
tables, and label the rows with the GPU name printed in section 1.
Expected wall clock: under 1 minute.

In [ ]:
print(f"Measured on {GPU_NAME}, repository commit {COMMIT}.")
print()
must([sys.executable, "scripts/results_table.py"], env=RUN_ENV)

## 7. CodeXGLUE reference recipe on Devign

The CodeXGLUE Defect Detection leaderboard number for CodeBERT (62.08% accuracy) comes
from a longer recipe than the reduced run in section 5: 5 epochs, 400 tokens, batch 32.
The CPU host could not fit that; a T4 can. This cell runs it through the same
`train_devign` module, changing only the environment variables the module already reads,
and freezes it separately as `results/devign_codebert_reference_recipe/` so it is never
confused with the section 5 row.

Whatever accuracy comes out is the accuracy that gets reported, including if it lands
below 62.08.

If a free-tier T4 runs out of memory at batch 32 and 400 tokens, drop `BATCH_SIZE` to 16
and say so beside the number; the recipe is then no longer the published one.
Expected wall clock: 20 to 40 minutes on a T4.

In [ ]:
train(
    "devign_codebert_reference_recipe",
    "ml_inference.train_devign",
    epochs=5,
    report="bench/reports/devign_metrics.json",
    extra_env={"BATCH_SIZE": "32", "MAX_SEQ_LEN": "400"},
)

ref = json.loads((REPO / "results/devign_codebert_reference_recipe/metrics.json").read_text())
print()
print("| System | Accuracy | Binary F1 |")
print("|---|---|---|")
print("| CodeBERT, CodeXGLUE leaderboard | 0.6208 | not reported |")
print(f"| this run on {GPU_NAME} | {ref['accuracy']:.4f} | {ref['f1_binary']:.4f} |")
delta = ref["accuracy"] - 0.6208
print()
print(f"difference against the published leaderboard: {delta:+.4f} accuracy")

## 8. Environment and artifact zip

Writes the GPU environment beside the results so every frozen run carries the host it was
measured on, then zips `results/` for download. Commit the unzipped contents to keep the
numbers checkable.
Expected wall clock: under 1 minute.

In [ ]:
env_path = REPO / "results" / "gpu_env.json"
env_path.write_text(
    json.dumps(
        {
            "gpu_name": GPU_NAME,
            "gpu_memory_gb": round(props.total_memory / 1e9, 2),
            "compute_capability": f"sm{props.major}{props.minor}",
            "cuda": torch.version.cuda,
            "torch": torch.__version__,
            "commit": COMMIT,
            "gates": [{"gate": g, "command": c, "seconds": round(s, 1)} for g, c, s in GATE_RESULTS],
        },
        indent=2,
    )
    + "\n"
)
print("environment:", env_path)

zip_path = BASE / "ml_inference_results_gpu.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(p for p in (REPO / "results").rglob("*") if p.is_file()):
        zf.write(file, file.relative_to(REPO))
print("results zip:", zip_path)